In [0]:
CREATE OR REPLACE TABLE workspace.default.zip_summary_v2 AS
WITH rat_stats AS (
  SELECT
    zip_code,
    MAX(borough) AS borough,

    COUNT(*) AS total_311_rodent_record_count,

    COUNT_IF(resident_report)
      AS resident_report_count,

    COUNT_IF(resident_rat_report)
      AS resident_rat_report_count,

    COUNT_IF(direct_rat_sighting)
      AS direct_rat_sighting_count,

    COUNT_IF(attracting_condition_report)
      AS attracting_condition_report_count,

    COUNT_IF(resident_mouse_report)
      AS resident_mouse_report_count,

    COUNT_IF(inspector_signs_record)
      AS inspector_signs_record_count,

    COUNT_IF(
      resident_rat_report
      AND closed_date IS NULL
    ) AS resident_rat_not_closed_count,

    COUNT_IF(
      resident_rat_report
      AND closed_date IS NOT NULL
    ) AS resident_rat_closed_count,

    COUNT_IF(
      resident_rat_report
      AND closed_within_60_seconds
    ) AS resident_rat_closed_within_60s_count,

    COUNT(
      DISTINCT CASE
        WHEN resident_rat_report
        THEN coordinate_key
      END
    ) AS resident_rat_distinct_coordinate_count

  FROM workspace.default.rat_clean_v2
  GROUP BY zip_code
),

inspection_stats AS (
  SELECT
    zip_code,
    MAX(borough) AS borough,

    COUNT(*) AS violation_row_count,

    COUNT(DISTINCT restaurant_id)
      AS distinct_restaurant_count,

    COUNT(DISTINCT inspection_id)
      AS distinct_inspection_count,

    COUNT(
      DISTINCT CASE
        WHEN rat_violation
        THEN restaurant_id
      END
    ) AS restaurants_with_rat_violation,

    COUNT(
      DISTINCT CASE
        WHEN mouse_violation
        THEN restaurant_id
      END
    ) AS restaurants_with_mouse_violation,

    COUNT(
      DISTINCT CASE
        WHEN rodent_violation
        THEN restaurant_id
      END
    ) AS restaurants_with_rodent_violation,

    COUNT(
      DISTINCT CASE
        WHEN rodent_violation
        THEN inspection_id
      END
    ) AS inspections_with_rodent_violation

  FROM workspace.default.inspections_clean_v2
  GROUP BY zip_code
)

SELECT
  COALESCE(r.zip_code, i.zip_code)
    AS zip_code,

  COALESCE(r.borough, i.borough)
    AS borough,

  COALESCE(r.total_311_rodent_record_count, 0)
    AS total_311_rodent_record_count,

  COALESCE(r.resident_report_count, 0)
    AS resident_report_count,

  COALESCE(r.resident_rat_report_count, 0)
    AS resident_rat_report_count,

  COALESCE(r.direct_rat_sighting_count, 0)
    AS direct_rat_sighting_count,

  COALESCE(r.attracting_condition_report_count, 0)
    AS attracting_condition_report_count,

  COALESCE(r.resident_mouse_report_count, 0)
    AS resident_mouse_report_count,

  COALESCE(r.inspector_signs_record_count, 0)
    AS inspector_signs_record_count,

  COALESCE(r.resident_rat_not_closed_count, 0)
    AS resident_rat_not_closed_count,

  COALESCE(r.resident_rat_closed_count, 0)
    AS resident_rat_closed_count,

  COALESCE(r.resident_rat_closed_within_60s_count, 0)
    AS resident_rat_closed_within_60s_count,

  COALESCE(r.resident_rat_distinct_coordinate_count, 0)
    AS resident_rat_distinct_coordinate_count,

  COALESCE(i.violation_row_count, 0)
    AS violation_row_count,

  COALESCE(i.distinct_restaurant_count, 0)
    AS distinct_restaurant_count,

  COALESCE(i.distinct_inspection_count, 0)
    AS distinct_inspection_count,

  COALESCE(i.restaurants_with_rat_violation, 0)
    AS restaurants_with_rat_violation,

  COALESCE(i.restaurants_with_mouse_violation, 0)
    AS restaurants_with_mouse_violation,

  COALESCE(i.restaurants_with_rodent_violation, 0)
    AS restaurants_with_rodent_violation,

  COALESCE(i.inspections_with_rodent_violation, 0)
    AS inspections_with_rodent_violation

FROM rat_stats r
FULL OUTER JOIN inspection_stats i
  ON r.zip_code = i.zip_code;

SELECT
  zip_code,
  borough,
  total_311_rodent_record_count,
  resident_rat_report_count,
  direct_rat_sighting_count,
  attracting_condition_report_count,
  inspector_signs_record_count,
  distinct_restaurant_count,
  distinct_inspection_count
FROM workspace.default.zip_summary_v2
WHERE zip_code IN (
  '10035',
  '11379',
  '11415',
  '11423',
  '11430'
)
ORDER BY zip_code;

WITH thresholds AS (
  SELECT *
  FROM VALUES
    (5),
    (50),
    (100),
    (200)
  AS t(minimum_resident_rat_reports)
)

SELECT
  t.minimum_resident_rat_reports,
  COUNT_IF(
    z.resident_rat_report_count
      >= t.minimum_resident_rat_reports
    AND z.distinct_restaurant_count >= 20
  ) AS scoreable_zip_count
FROM workspace.default.zip_summary_v2 z
CROSS JOIN thresholds t
GROUP BY t.minimum_resident_rat_reports
ORDER BY t.minimum_resident_rat_reports;

SELECT
  zip_code,
  borough,
  resident_rat_report_count,
  distinct_restaurant_count,

  CASE
    WHEN distinct_restaurant_count < 20
      THEN 'Insufficient restaurant sample'
    WHEN resident_rat_report_count < 200
      THEN 'Insufficient resident-report sample'
    ELSE 'Scoreable'
  END AS sample_status

FROM workspace.default.zip_summary_v2
WHERE resident_rat_report_count < 200
ORDER BY
  resident_rat_report_count ASC,
  zip_code;
